# Graphic design 01

Macro idea → designer’s proposal and image prompt → generated poster → same designer
examines the poster → revises its proposal or prompt → next poster.

The designer judges whether visible choices communicate the macro idea, using semiotic
reasoning where helpful. Its own proposal is provisional. No independent critic is required.


## 1. Setup

Launch Jupyter from the project directory. Install `requirements.txt` first.
Load local configuration without displaying secrets. Existing process variables take priority,
then `.env.local`, then `.env`. The generation cell runs a paid API call only when `RUN_GENERATION` is set to `True`.


In [ ]:
import os
import json
from pathlib import Path
from dotenv import dotenv_values
from copy import deepcopy
from IPython.display import Markdown, display
from workflow import generate_round, save_round
from images import compare_images
from records import save_experiment

ROOT = Path.cwd()
if not (ROOT / "graphic_design_01.ipynb").is_file():
    raise RuntimeError("Launch Jupyter from the graphic-design-helper directory.")

local_config = {
    **dotenv_values(ROOT / ".env"),
    **dotenv_values(ROOT / ".env.local"),
}
for key, value in local_config.items():
    if value is not None:
        os.environ.setdefault(key, value)
del local_config
from design import designer_prompt, review_token
from proposals import propose_design, PROPOSAL_SCHEMA


## 2. Macro brief: our input

Supply purpose, audience, encounter setting, tone, and genuine deliverable constraints.
The model proposes the concept, sign relationships, typography, color, composition, and wording.
No preselected motifs or manually assigned sign categories enter its initial request.
The earlier headline is retained below as an optional reference only, not sent to the designer.


In [ ]:
brief = {
    "topic": "Slowing down in everyday life",
    "purpose": "Invite people to reconsider pressure to remain continuously productive and allow a brief pause.",
    "audience": "Provisional: students or office workers feeling pressure to stay productive.",
    "setting": "A poster encountered briefly in a shared indoor space.",
    "tone": "Inviting rather than blaming or patronizing.",
    "constraints": "Avoid unsupported factual, health, or society-wide claims.",
    "deliverable": "A portrait poster in English. Propose the visible wording.",
    "exact_copy": None,  # Supply text here only if it is a genuine requirement.
}
previous_headline = "A moment, without a project."  # Not included in the request.
if "rounds" not in globals():
    rounds = []
if "proposal_records" not in globals():
    proposal_records = []


## 3. Preview the designer request

For the first poster, the designer receives the macro brief. After generation, rerun this
section: the same designer receives the actual poster, its previous proposal, and the exact
prompt used. In one call it examines whether the poster communicates the macro idea and
returns a revised proposal or prompt. `revision_summary` explains its observations and changes.

The image generator receives only `production_prompt`; the designer's analysis stays separate.


In [ ]:
designer_model = os.getenv("OPENAI_DESIGNER_MODEL", "").strip() or "gpt-5.6-luna"
designer_effort = "medium"
revision_context = None
designer_image = None
if rounds:
    previous = rounds[-1]
    if brief != previous["research"]["brief"]:
        raise ValueError("The macro brief changed. Start a new experiment with rounds = [].")
    if not previous.get("image_path"):
        raise ValueError("The last attempt has no image. Resolve that attempt before continuing.")
    revision_context = {
        "parent_folder": previous["folder"],
        "previous_proposal": previous["research"].get("proposal"),
        "previous_prompt": previous["production_prompt"],
    }
    designer_image = previous["image_path"]
    designer_model = previous["research"]["designer_record"]["model"]
    display(compare_images([designer_image], ["Poster for designer self-review"]))
designer_input = deepcopy({"brief": brief, "revision": revision_context})
designer_draft = designer_prompt(**designer_input)
print(designer_draft)
print({"model": designer_model, "reasoning_effort": designer_effort,
       "image": designer_image})


## 4. Designer: propose, or examine and revise

Set `RUN_DESIGNER` to `True` to make one paid call. On later rounds, this is both self-review
and revision: read `revision_summary` for visible evidence, communication problems, and why
changes should help. The complete updated proposal follows. If it finds no justified change,
stop here and keep the current poster. No image generation is triggered by this call.


In [ ]:
# RUN_DESIGNER = False  # Set to True to call the designer and propose a design.
RUN_DESIGNER = False
if RUN_DESIGNER:
    if brief != designer_input["brief"] or designer_draft != designer_prompt(**designer_input):
        raise ValueError("The inputs changed. Preview the designer request again.")
    if rounds and (not revision_context or
                   revision_context["parent_folder"] != rounds[-1]["folder"] or
                   designer_image != rounds[-1].get("image_path")):
        raise ValueError("Preview the current poster for designer self-review first.")
    proposal_record = propose_design(
        designer_draft, model=designer_model, reasoning_effort=designer_effort,
        image_path=designer_image, output_dir=ROOT / "outputs",
    )
    proposal_record["designer_input"] = deepcopy(designer_input)
    (Path(proposal_record["folder"]) / "proposal.json").write_text(
        json.dumps(proposal_record, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    proposal_records.append(deepcopy(proposal_record))
    if rounds:
        rounds[-1]["self_review"] = deepcopy(proposal_record)
        save_round(rounds[-1])
    print(json.dumps(proposal_record["proposal"], ensure_ascii=False, indent=2))
else:
    print("Designer call is off. Preview the request, then enable when ready.")


## 5. Review the proposed production prompt

Only the proposal's `production_prompt` is sent to GPT Image. The complete proposal remains
available to the designer and experiment record. You may edit the draft before marking it reviewed;
the original model proposal is preserved separately. Any brief or proposal change requires review.


In [ ]:
production_prompt = proposal_record["proposal"]["production_prompt"] if "proposal_record" in globals() else ""
image_settings = {
    "model": os.getenv("OPENAI_IMAGE_MODEL", "gpt-image-2"),
    "size": "1024x1536", "quality": "medium",
}
reviewed_token = None

def current_research():
    if "proposal_record" not in globals():
        raise ValueError("Request a design proposal first.")
    if brief != proposal_record["designer_input"]["brief"]:
        raise ValueError("The proposal belongs to a different brief. Request a new proposal.")
    return deepcopy({"brief": brief, "proposal": proposal_record["proposal"],
                     "designer_record": proposal_record})

print(production_prompt or "No production prompt available yet.")
print(image_settings)


In [ ]:
# MARK_PROMPT_REVIEWED = False
MARK_PROMPT_REVIEWED = False

if MARK_PROMPT_REVIEWED:
    reviewed_token = review_token(production_prompt, current_research(), image_settings)
    print("Current prompt, rationale, and settings marked reviewed.")


## 6. Generate the poster

Enable one generation after inspecting the proposed prompt. This experiment retains the
limit of one initial attempt and two revisions. Each request saves its proposal, prompt and image.
The next poster is generated from the updated text prompt; this notebook does not edit the
previous image's pixels. The previous image is visual input to the designer's self-review.


In [ ]:
# RUN_GENERATION = False
RUN_GENERATION = False

if RUN_GENERATION:
    research = current_research()
    context = proposal_record["designer_input"]["revision"]
    if rounds and (not context or context["parent_folder"] != rounds[-1]["folder"]):
        raise ValueError("Have the designer examine the current poster and revise first.")
    revision_decision = {
        "reason": research["proposal"]["revision_summary"] if rounds else "Initial proposal",
    }
    active_round = generate_round(
        production_prompt, research, image_settings,
        approved_token=reviewed_token, history=rounds, revision=revision_decision,
        output_dir=ROOT / "outputs",
    )
    reviewed_token = None
    print(f"Saved round {active_round['round']} to {active_round['folder']}")
else:
    print("Image generation is off.")


In [ ]:
image_paths = [r["image_path"] for r in rounds if r.get("image_path")]
labels = [f"Round {r['round']}" for r in rounds if r.get("image_path")]
if image_paths:
    display(compare_images(image_paths, labels))


## 7. Look, reconsider, repeat

Return to **Section 3**, then run Sections 3–6 for the next iteration. The designer sees the
latest poster directly and asks: does this communicate the macro idea? It may revise the
underlying proposal or simply correct the production prompt. Keep successful choices.

Read the self-review before deciding to generate again. Stop when no useful change remains
or the attempt limit is reached; reaching the limit does not establish success. A model's
reading is a design judgment, not evidence of audience reception.

After a kernel restart, you can resume by loading your experiment's ordered `round.json`
files into `rounds`, then returning to Section 3. Earlier saved rounds can be self-reviewed
without rerunning their old critic stages. Pilot 1 and its displayed outputs are archived in `pilot1/graphic_design_01.ipynb`.
Its generated files are in `pilot1/outputs/`; `pilot1/index.json` lists their current paths.
Archived records retain their original run-time paths as provenance.


## 8. Optional audience readings

Before explaining the intention, ask: What did you notice first? What do you think this is saying?
Who seems to be speaking? Does it invite you to do anything? Record unexpected readings too.
These are exploratory observations, not evidence of long-term behavior change.

Store participants' words separately from your interpretation. Avoid identifying information.

For each element, compare the intended relationship with the reading people actually describe.
Ask about the imagery and arrangement in ordinary language; viewers need not know Peircean terms.
If something looks like evidence, ask what they think it documents. Do not supply that answer first.


In [ ]:
observations = []  # For each response: variant, anonymous label, verbatim wording, researcher interpretation.
reflection = {
    "what_worked": "",
    "unexpected_readings": "",
    "design_changes_to_try": "",
    "semiotic_assumptions_to_revisit": "",
    "relations_supported_or_challenged_by_responses": "",
    "source_or_evidence_confusions": "",
    "limitations": "No audience responses recorded in this reflection.",
}


## 9. Save a snapshot

Set `SAVE_SNAPSHOT` to `True` when you want to save the current state. Each save creates a new folder
with notes, image copies, and hashes. Include only non-secret settings; never pass environment
variables, API keys, or client objects. The saved status distinguishes planning from collected evidence.


In [ ]:
SAVE_SNAPSHOT = False
if SAVE_SNAPSHOT:
    run = save_experiment({
        "status": "exploration" if rounds else "planning",
        "brief": brief, "designer_prompt": designer_draft,
        "proposal_records": proposal_records,
        "production_prompt_draft": production_prompt, "rounds": rounds,
        "audience_observations": observations, "reflection": reflection,
    }, [r["image_path"] for r in rounds if r.get("image_path")], ROOT / "outputs")
    print(f"Saved: {run}")
